# Lesson 9.6 - Speech/Audio AI and Voice Agents Demo

## Learning Objectives
- Simulate a voice-agent loop with ASR, policy, and response stages.
- Measure turn-level latency budgets.
- Design safe action gating for spoken commands.

## Educational Scope and Production Extension Note
This notebook intentionally uses lightweight simulation/stub components for teaching. Treat outputs as instructional, not production-ready.

For production: replace simulated components with real services/models, add reliability controls, and instrument full observability.

In [1]:
from __future__ import annotations

import time
from dataclasses import dataclass

import numpy as np
import pandas as pd

## 1) Simulate Audio Front-End and ASR Output

In [2]:
sample_rate = 16000
seconds = 2
wave = 0.1 * np.random.randn(sample_rate * seconds)
print({"samples": wave.shape[0], "duration_s": seconds})


def asr_stub(audio: np.ndarray) -> str:
    # In real systems this would be Whisper/wav2vec-style inference.
    return "schedule maintenance window for payment api at 2 am"

transcript = asr_stub(wave)
transcript

{'samples': 32000, 'duration_s': 2}


'schedule maintenance window for payment api at 2 am'

## 2) Intent Parsing + Safety Gate

In [3]:
@dataclass
class ActionDecision:
    intent: str
    requires_confirmation: bool
    safe_to_execute: bool


def policy_engine(text: str) -> ActionDecision:
    text = text.lower()
    if "delete" in text or "shutdown" in text:
        return ActionDecision("high_risk_action", True, False)
    if "schedule" in text and "maintenance" in text:
        return ActionDecision("schedule_maintenance", True, True)
    return ActionDecision("qa_response", False, True)

decision = policy_engine(transcript)
decision

ActionDecision(intent='schedule_maintenance', requires_confirmation=True, safe_to_execute=True)

## 3) Response Generation + TTS Stub

In [4]:
def response_generator(decision: ActionDecision) -> str:
    if decision.intent == "schedule_maintenance":
        return "I can schedule that maintenance window. Please confirm service and timezone."
    if decision.intent == "high_risk_action":
        return "I cannot execute that request without explicit approval."
    return "I can help with that request. Could you provide more context?"

def tts_stub(text: str) -> np.ndarray:
    # Fake synthesized waveform length proportional to text length.
    return np.zeros(len(text) * 120, dtype=np.float32)

response_text = response_generator(decision)
voice_out = tts_stub(response_text)
print(response_text)
print({"tts_samples": int(voice_out.shape[0])})

I can schedule that maintenance window. Please confirm service and timezone.
{'tts_samples': 9120}


## 4) Turn Latency Budget Monitoring

In [5]:
def timed_stage(name, fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return {"stage": name, "ms": (t1 - t0) * 1000, "output": out}

results = []
results.append(timed_stage("asr", asr_stub, wave))
results.append(timed_stage("policy", policy_engine, results[-1]["output"]))
results.append(timed_stage("nlg", response_generator, results[-1]["output"]))
results.append(timed_stage("tts", tts_stub, results[-1]["output"]))

lat = pd.DataFrame([{"stage": r["stage"], "ms": r["ms"]} for r in results])
lat

,stage,ms
0,asr,0.000420
1,policy,0.003877
2,nlg,0.000481
3,tts,0.005721


## 5) Optional Hugging Face ASR Hook (if model + deps available)

In [6]:
ENABLE_HF_ASR = False

if ENABLE_HF_ASR:
    from transformers import pipeline
    transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
    # transcriber("path_to_audio.wav")
    print("HF ASR pipeline is configured.")
else:
    print("Skipped HF ASR run (set ENABLE_HF_ASR=True to enable).")

Skipped HF ASR run (set ENABLE_HF_ASR=True to enable).


## Connect to Theory
- ASR quality and latency drive downstream performance.
- Safety policy must gate high-risk intents before tool execution.
- Turn latency should be monitored at stage level.

## Production Extension Checklist
- Replace simulated/stub components with production services or managed endpoints.
- Add structured logging, tracing IDs, and latency/error dashboards.
- Add input/output validation and policy/safety guardrails.
- Add retry/timeouts, rate limiting, and fallback behavior.
- Add offline/online evaluation gates before release.
- Add secrets management and environment-specific configuration.
- Add CI checks and smoke tests for critical paths.

## Case Studies & Exceptions
1. Contact-center copilot: streaming ASR + retrieval helps agents respond faster.
2. Clinical dictation: low-confidence segments require human validation.
3. Exception: very noisy environments may require non-voice fallback UI.

## Interview Questions & Answers
1. **Q:** Main ASR metric? **A:** WER plus latency metrics.
2. **Q:** Why stage latency? **A:** It isolates bottlenecks across ASR/policy/TTS.
3. **Q:** What is barge-in? **A:** User interrupting agent speech.
4. **Q:** Why confirmation gates? **A:** To prevent unsafe/irreversible actions.
5. **Q:** Cloud vs edge voice trade-off? **A:** Capability vs privacy/offline behavior.
6. **Q:** ASR bottleneck impact? **A:** Transcript errors propagate to intent and response.
7. **Q:** Why need VAD? **A:** To segment speech and reduce compute noise.
8. **Q:** Voice-specific safety risk? **A:** Spoken prompt injection/impersonation attacks.
9. **Q:** How to handle low confidence? **A:** Clarify, fallback, or human handoff.
10. **Q:** One-line design rule? **A:** Optimize full turn quality, not isolated model scores.